
# BKK 3-Class V5 IMERG-Observed Labels And Satellite History Features

V5 keeps the V4 strong-first, recall-constrained hierarchy but changes what the model is
trained to predict and what it is allowed to see.

## What changed from V4

| | V4 | V5 |
| --- | --- | --- |
| Label source | `OM_BKK_DATA_PRECOMPUTE.precipitation_next_{h}h` (ECMWF model output) | `IMERG_BKK_DATA.precipitation_mm` at `t+h` (GPM satellite observation) |
| Rain history features | Open-Meteo precipitation lags only | Open-Meteo lags **plus** IMERG observed lags, rolling sums, neighbour aggregates |
| Model-vs-observation bias | not represented | explicit `om_minus_imerg_*` features |
| Boosting rounds | fixed 500 | early stopping on the validation split |
| Threshold fallback | max recall when the constraint is unreachable, which collapses to predicting everything strong | max strong F1, with the miss still reported |
| Threshold grid | strong threshold capped at 0.61 | searched to 0.95 |
| Baseline | none | IMERG persistence baseline for every horizon |
| Data provenance | not tracked | test metrics broken out by IMERG `run_type` |

## Why the label change matters

`IMERG_BKK_DATA` now covers 2021-07-20 to 2026-07-20 for all 56 Bangkok grid cells,
43,823 complete hours per cell, with no missing values. Joined hour-for-hour against
`OM_BKK_DATA_PRECOMPUTE` on `(grid_number, local time)` it produces 2,453,696 matched rows.

Two facts from that join motivate V5:

1. **Correlation between the two precipitation series is only 0.18.** The Open-Meteo
   precipitation field that V4 used as ground truth agrees only weakly with what the
   satellite actually observed.
2. **The diurnal cycles are ~5 hours out of phase.** Open-Meteo peaks at 15:00 local
   (0.475 mm/h mean); IMERG peaks at 20:00 local (0.302 mm/h mean). This is the well
   documented early-convection timing bias of global NWP over the tropics.

A V4 model that scores well is therefore partly reproducing ECMWF's timing error. Section 5
of this notebook reproduces both facts from the loaded data so the change is auditable
rather than asserted.

## Operational caveat you must decide on

IMERG Final Run has roughly a 3.5-month latency, so `imerg_*` features at time `t` are not
available at time `t` in real-time operation. Two settings control this:

- `USE_IMERG_HISTORY_FEATURES` - set to `False` to train a deployable model that predicts
  IMERG-observed rain from Open-Meteo inputs alone.
- `IMERG_FEATURE_LAG_HOURS` - shift every IMERG feature back by N hours to emulate the
  latency of the IMERG Early Run (roughly 4 hours) or of a nowcast feed.

Leaving both at the research defaults (`True`, `0`) gives the upper bound on skill, not a
deployable number. Run the notebook twice and compare.

## Classes (unchanged)

- `0 = no_rain`: `< 0.1 mm`
- `1 = light`: `0.1 mm` to `< 2.5 mm`
- `2 = moderate_or_heavy`: `>= 2.5 mm`

## 1. Setup

In [ ]:

import json
import os
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 2. Configuration

In [ ]:

DB_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "postgres"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", "Pass1234"),
}

TABLE_NAME = '"OM_BKK_DATA"'
PRECOMPUTE_TABLE_NAME = '"OM_BKK_DATA_PRECOMPUTE"'
IMERG_TABLE_NAME = '"IMERG_BKK_DATA"'
GRID_TABLE_NAME = '"Bangkok_Grid_9km"'

PROJECT_ROOT = Path.cwd()
MODEL_DIR = PROJECT_ROOT / "ML_Model_V2" / "trained_models" / "BKK_3_Class_V5"
CACHE_DIR = PROJECT_ROOT / "ML_Model_V2" / "cache"

HORIZONS = [1, 2, 3, 4, 5, 6]
CLASS_LABELS = {
    0: "no_rain",
    1: "light",
    2: "moderate_or_heavy",
}
CLASS_ORDER = list(CLASS_LABELS)
INTENSITY_THRESHOLDS_MM = {
    "rain_min": 0.1,
    "moderate_or_heavy_min": 2.5,
}

# --- V5 label source -------------------------------------------------------
# "imerg"      -> classes come from GPM IMERG observed hourly accumulation (V5 default)
# "open_meteo" -> classes come from the Open-Meteo precipitation field (reproduces V4)
LABEL_SOURCE = "imerg"

# Also score the V5 model against the other label source, so V4 and V5 numbers are
# comparable on a like-for-like basis.
EVALUATE_BOTH_LABEL_SOURCES = True

# --- V5 feature switches ---------------------------------------------------
# IMERG Final Run lags ~3.5 months, so IMERG features are NOT available in real time.
# Set to False for a deployable Open-Meteo-only model that still predicts observed rain.
USE_IMERG_HISTORY_FEATURES = True

# Shift every IMERG feature back by this many hours to emulate a realistic feed latency.
# 0  = research upper bound (observation at t is visible at t)
# 4  = approximate IMERG Early Run latency
IMERG_FEATURE_LAG_HOURS = 0

# IMERG run types to include. "permanent" is Final Run; "provisional" is the Late Run
# that backs the most recent months (2025-10-01 onward) and therefore the whole test
# split. Dropping it removes the test period entirely, so it is kept by default and
# reported separately in section 11.
IMERG_RUN_TYPES = ("permanent", "provisional", "Final")

# --- tuning / training -----------------------------------------------------
# V4 capped the strong grid at 0.61, which makes high-precision operating points
# unreachable at the longer horizons where `class_weight="balanced"` inflates the
# probabilities. V5 searches the full range.
STRONG_PROBABILITY_THRESHOLDS_TO_TEST = np.arange(0.01, 0.96, 0.02)
LIGHT_PROBABILITY_THRESHOLDS_TO_TEST = np.arange(0.20, 0.86, 0.05)
TARGET_VALIDATION_STRONG_RECALL = 0.82
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
SAMPLE_ROWS = None
RANDOM_STATE = 42

N_ESTIMATORS = 1500
EARLY_STOPPING_ROUNDS = 60
LEARNING_RATE = 0.04

USE_CACHE = True

# The cache key carries every setting that changes the query, so flipping
# IMERG_FEATURE_LAG_HOURS or IMERG_RUN_TYPES cannot silently reuse a stale join.
CACHE_PATH = CACHE_DIR / (
    f"bkk_3_class_v5_joined_lag{IMERG_FEATURE_LAG_HOURS}h"
    f"_{'-'.join(sorted(IMERG_RUN_TYPES))}.pkl"
)

LABEL_PREFIXES = {"imerg": "imerg", "open_meteo": "om"}
assert LABEL_SOURCE in LABEL_PREFIXES, LABEL_SOURCE
LABEL_PREFIX = LABEL_PREFIXES[LABEL_SOURCE]

print(f"Label source              : {LABEL_SOURCE}")
print(f"IMERG history features    : {USE_IMERG_HISTORY_FEATURES}")
print(f"IMERG feature lag (hours) : {IMERG_FEATURE_LAG_HOURS}")
print(f"IMERG run types           : {IMERG_RUN_TYPES}")

## 3. Feature Columns

In [ ]:

BASELINE_FEATURE_COLUMNS = [
    "temperature_2m", "relative_humidity_2m", "pressure_msl", "surface_pressure",
    "dew_point_2m", "precipitation", "cloud_cover", "wind_speed_10m",
    "wind_direction_10m", "temperature_dew_point_spread", "pressure_msl_change_3h",
    "pressure_msl_change_6h", "precipitation_lag_1h", "precipitation_lag_2h",
    "precipitation_lag_3h", "precipitation_lag_6h", "precipitation_sum_past_3h",
    "precipitation_sum_past_6h", "precipitation_sum_past_12h", "precipitation_sum_past_24h",
    "cloud_cover_lag_1h", "cloud_cover_lag_3h", "cloud_cover_lag_6h",
    "humidity_lag_1h", "humidity_lag_3h", "humidity_lag_6h",
    "wind_speed_lag_1h", "wind_speed_lag_3h", "hour_sin", "hour_cos",
    "month_sin", "month_cos", "grid_row", "grid_column", "latitude", "longitude",
]

NEIGHBOR_FEATURE_COLUMNS = [
    "neighbor_count", "neighbor_precipitation_mean", "neighbor_precipitation_max",
    "neighbor_precipitation_sum", "neighbor_rain_count", "neighbor_rain_rate",
    "neighbor_cloud_cover_mean", "neighbor_cloud_cover_max", "neighbor_relative_humidity_mean",
    "neighbor_relative_humidity_max", "neighbor_pressure_msl_mean", "neighbor_pressure_msl_min",
    "neighbor_pressure_msl_max", "neighbor_temperature_2m_mean", "neighbor_dew_point_2m_mean",
    "neighbor_temperature_dew_point_spread_mean", "neighbor_wind_speed_10m_mean",
    "neighbor_wind_speed_10m_max", "row_minus_precipitation_mean", "row_plus_precipitation_mean",
    "column_minus_precipitation_mean", "column_plus_precipitation_mean", "row_minus_cloud_cover_mean",
    "row_plus_cloud_cover_mean", "column_minus_cloud_cover_mean", "column_plus_cloud_cover_mean",
    "neighbor_precipitation_mean_minus_center", "neighbor_cloud_cover_mean_minus_center",
    "neighbor_relative_humidity_mean_minus_center", "center_pressure_msl_minus_neighbor_mean",
]

# --- V5: observed satellite rain history -----------------------------------
IMERG_CENTER_FEATURE_COLUMNS = [
    "imerg_precipitation", "imerg_rate_max",
    "imerg_precipitation_lag_1h", "imerg_precipitation_lag_2h",
    "imerg_precipitation_lag_3h", "imerg_precipitation_lag_6h",
    "imerg_precipitation_sum_past_3h", "imerg_precipitation_sum_past_6h",
    "imerg_precipitation_sum_past_12h", "imerg_precipitation_sum_past_24h",
    "imerg_rate_max_past_6h", "imerg_rain_hours_past_6h", "imerg_rain_hours_past_24h",
    "imerg_precipitation_change_1h", "imerg_precipitation_change_3h",
]

IMERG_NEIGHBOR_FEATURE_COLUMNS = [
    "imerg_neighbor_precipitation_mean", "imerg_neighbor_precipitation_max",
    "imerg_neighbor_precipitation_sum", "imerg_neighbor_rain_rate",
    "imerg_neighbor_rate_max", "imerg_neighbor_precipitation_mean_minus_center",
]

# Model-minus-observation disagreement: tells the model whether ECMWF is currently
# running wet or dry over this cell, which V4 had no way to express.
IMERG_BIAS_FEATURE_COLUMNS = [
    "om_minus_imerg_precipitation",
    "om_minus_imerg_sum_past_6h",
    "om_minus_imerg_sum_past_24h",
]

IMERG_FEATURE_COLUMNS = (
    IMERG_CENTER_FEATURE_COLUMNS
    + IMERG_NEIGHBOR_FEATURE_COLUMNS
    + IMERG_BIAS_FEATURE_COLUMNS
)

OPEN_METEO_FEATURE_COLUMNS = BASELINE_FEATURE_COLUMNS + NEIGHBOR_FEATURE_COLUMNS
FEATURE_COLUMNS = list(OPEN_METEO_FEATURE_COLUMNS)
if USE_IMERG_HISTORY_FEATURES:
    FEATURE_COLUMNS = FEATURE_COLUMNS + IMERG_FEATURE_COLUMNS

FEATURE_SET_NAME = "neighbor_grid_imerg" if USE_IMERG_HISTORY_FEATURES else "neighbor_grid"

IMERG_FUTURE_COLUMNS = [f"imerg_precipitation_next_{h}h" for h in HORIZONS]
OM_FUTURE_COLUMNS = [f"om_precipitation_next_{h}h" for h in HORIZONS]
FUTURE_PRECIP_COLUMNS = IMERG_FUTURE_COLUMNS + OM_FUTURE_COLUMNS

FINAL_TARGET_COLUMNS = [f"rain_3class_next_{h}h" for h in HORIZONS]
STRONG_TARGET_COLUMNS = [f"strong_vs_rest_next_{h}h" for h in HORIZONS]
LIGHT_TARGET_COLUMNS = [f"light_vs_no_rain_next_{h}h" for h in HORIZONS]
TARGET_COLUMNS = FINAL_TARGET_COLUMNS + STRONG_TARGET_COLUMNS + LIGHT_TARGET_COLUMNS

print(f"Open-Meteo features : {len(OPEN_METEO_FEATURE_COLUMNS)}")
print(f"IMERG features      : {len(IMERG_FEATURE_COLUMNS) if USE_IMERG_HISTORY_FEATURES else 0}")
print(f"Total features used : {len(FEATURE_COLUMNS)}")
print(f"Feature set name    : {FEATURE_SET_NAME}")
print(f"Final 3-class targets: {FINAL_TARGET_COLUMNS}")


## 4. Load Joined Open-Meteo + IMERG Features

The query joins `OM_BKK_DATA_PRECOMPUTE` to `IMERG_BKK_DATA` on `(grid_number, local hour)`
and derives the IMERG lag, rolling and neighbour features with window functions. Two
separate joins into the IMERG window CTE are used when `IMERG_FEATURE_LAG_HOURS > 0`, so
features can be read from `t - lag` while labels are always read from `t`.

`imerg_history_hours_24 = 24` drops the partial rolling windows at the start of each grid
cell's series, so `imerg_precipitation_sum_past_24h` is never a short sum.

The full join takes roughly two minutes, so the result is cached under `ML_Model_V2/cache/`.
The cache filename encodes `IMERG_FEATURE_LAG_HOURS` and `IMERG_RUN_TYPES`, so changing
either setting cannot silently reuse a stale join. Set `USE_CACHE = False` to force a
reload. The full five-year cache is roughly 1 GB, and this repository has no `.gitignore`,
so add `ML_Model_V2/cache/` to one before committing.

In [ ]:

def connect():
    return psycopg2.connect(**DB_CONFIG)


def _window_frame(offset, hours):
    """ROWS frame covering `hours` ending `offset` rows before the current row."""
    start = offset + hours - 1
    end = "CURRENT ROW" if offset == 0 else f"{offset} PRECEDING"
    return f"ROWS BETWEEN {start} PRECEDING AND {end}"


def _lagged(expression, offset):
    return expression if offset == 0 else f"LAG({expression}, {offset}) OVER w"


def build_query(imerg_feature_lag_hours=0, run_types=IMERG_RUN_TYPES):
    offset = int(imerg_feature_lag_hours)
    rain_min = INTENSITY_THRESHOLDS_MM["rain_min"]
    run_type_list = ", ".join(f"'{r}'" for r in run_types)

    lead_sql = ",\n        ".join(
        f"LEAD(precipitation_mm, {h}) OVER w AS imerg_precipitation_next_{h}h" for h in HORIZONS
    )
    om_lead_sql = ",\n    ".join(
        f"p.precipitation_next_{h}h AS om_precipitation_next_{h}h" for h in HORIZONS
    )
    imerg_label_sql = ",\n    ".join(
        f"ilab.imerg_precipitation_next_{h}h" for h in HORIZONS
    )
    om_baseline_sql = ",\n    ".join(f"p.{column}" for column in OPEN_METEO_FEATURE_COLUMNS)

    # Feature-side and label-side aliases into the same window CTE.
    feature_alias = "ilab" if offset == 0 else "ifeat"
    feature_join = (
        ""
        if offset == 0
        else f"""JOIN imerg_window ifeat
      ON ifeat.grid_number = p.grid_number
     AND ifeat.t = p.local_forecast_time - INTERVAL '{offset} hour'"""
    )
    neighbor_time = (
        "p.local_forecast_time"
        if offset == 0
        else f"p.local_forecast_time - INTERVAL '{offset} hour'"
    )

    imerg_feature_sql = f"""
    {feature_alias}.imerg_precipitation,
    {feature_alias}.imerg_rate_max,
    {feature_alias}.imerg_precipitation_lag_1h,
    {feature_alias}.imerg_precipitation_lag_2h,
    {feature_alias}.imerg_precipitation_lag_3h,
    {feature_alias}.imerg_precipitation_lag_6h,
    {feature_alias}.imerg_precipitation_sum_past_3h,
    {feature_alias}.imerg_precipitation_sum_past_6h,
    {feature_alias}.imerg_precipitation_sum_past_12h,
    {feature_alias}.imerg_precipitation_sum_past_24h,
    {feature_alias}.imerg_rate_max_past_6h,
    {feature_alias}.imerg_rain_hours_past_6h,
    {feature_alias}.imerg_rain_hours_past_24h,
    {feature_alias}.imerg_precipitation - {feature_alias}.imerg_precipitation_lag_1h
        AS imerg_precipitation_change_1h,
    {feature_alias}.imerg_precipitation - {feature_alias}.imerg_precipitation_lag_3h
        AS imerg_precipitation_change_3h,
    nb.imerg_neighbor_precipitation_mean,
    nb.imerg_neighbor_precipitation_max,
    nb.imerg_neighbor_precipitation_sum,
    nb.imerg_neighbor_rain_rate,
    nb.imerg_neighbor_rate_max,
    nb.imerg_neighbor_precipitation_mean - {feature_alias}.imerg_precipitation
        AS imerg_neighbor_precipitation_mean_minus_center,
    p.precipitation - {feature_alias}.imerg_precipitation AS om_minus_imerg_precipitation,
    p.precipitation_sum_past_6h - {feature_alias}.imerg_precipitation_sum_past_6h
        AS om_minus_imerg_sum_past_6h,
    p.precipitation_sum_past_24h - {feature_alias}.imerg_precipitation_sum_past_24h
        AS om_minus_imerg_sum_past_24h,"""

    return f"""
WITH grid_pairs AS (
    SELECT a.grid_number AS center_grid, b.grid_number AS neighbor_grid
    FROM {GRID_TABLE_NAME} a
    JOIN {GRID_TABLE_NAME} b
      ON ABS(a.grid_row - b.grid_row) <= 1
     AND ABS(a.grid_column - b.grid_column) <= 1
     AND a.grid_number <> b.grid_number
), imerg_base AS (
    SELECT
        grid_number,
        local_observation_time AS t,
        precipitation_mm,
        precipitation_rate_max_mm_h,
        run_type
    FROM {IMERG_TABLE_NAME}
    WHERE is_complete_hour
      AND run_type IN ({run_type_list})
), imerg_window AS (
    SELECT
        grid_number,
        t,
        run_type,
        precipitation_mm AS imerg_precipitation,
        precipitation_rate_max_mm_h AS imerg_rate_max,
        LAG(precipitation_mm, 1) OVER w AS imerg_precipitation_lag_1h,
        LAG(precipitation_mm, 2) OVER w AS imerg_precipitation_lag_2h,
        LAG(precipitation_mm, 3) OVER w AS imerg_precipitation_lag_3h,
        LAG(precipitation_mm, 6) OVER w AS imerg_precipitation_lag_6h,
        SUM(precipitation_mm) OVER (PARTITION BY grid_number ORDER BY t {_window_frame(0, 3)})
            AS imerg_precipitation_sum_past_3h,
        SUM(precipitation_mm) OVER (PARTITION BY grid_number ORDER BY t {_window_frame(0, 6)})
            AS imerg_precipitation_sum_past_6h,
        SUM(precipitation_mm) OVER (PARTITION BY grid_number ORDER BY t {_window_frame(0, 12)})
            AS imerg_precipitation_sum_past_12h,
        SUM(precipitation_mm) OVER (PARTITION BY grid_number ORDER BY t {_window_frame(0, 24)})
            AS imerg_precipitation_sum_past_24h,
        MAX(precipitation_rate_max_mm_h) OVER (PARTITION BY grid_number ORDER BY t {_window_frame(0, 6)})
            AS imerg_rate_max_past_6h,
        SUM(CASE WHEN precipitation_mm >= {rain_min} THEN 1.0 ELSE 0.0 END)
            OVER (PARTITION BY grid_number ORDER BY t {_window_frame(0, 6)})
            AS imerg_rain_hours_past_6h,
        SUM(CASE WHEN precipitation_mm >= {rain_min} THEN 1.0 ELSE 0.0 END)
            OVER (PARTITION BY grid_number ORDER BY t {_window_frame(0, 24)})
            AS imerg_rain_hours_past_24h,
        COUNT(precipitation_mm) OVER (PARTITION BY grid_number ORDER BY t {_window_frame(0, 24)})
            AS imerg_history_hours_24,
        {lead_sql}
    FROM imerg_base
    WINDOW w AS (PARTITION BY grid_number ORDER BY t)
), imerg_neighbor AS (
    SELECT
        gp.center_grid AS grid_number,
        n.t,
        AVG(n.precipitation_mm) AS imerg_neighbor_precipitation_mean,
        MAX(n.precipitation_mm) AS imerg_neighbor_precipitation_max,
        SUM(n.precipitation_mm) AS imerg_neighbor_precipitation_sum,
        AVG(CASE WHEN n.precipitation_mm >= {rain_min} THEN 1.0 ELSE 0.0 END)
            AS imerg_neighbor_rain_rate,
        MAX(n.precipitation_rate_max_mm_h) AS imerg_neighbor_rate_max
    FROM grid_pairs gp
    JOIN imerg_base n ON n.grid_number = gp.neighbor_grid
    GROUP BY 1, 2
)
SELECT
    p.grid_number,
    p.local_forecast_time AS forecast_time,
    ilab.run_type AS imerg_run_type,
    {om_baseline_sql},
    {om_lead_sql},
    {imerg_label_sql},
    {imerg_feature_sql}
    p.precipitation AS om_precipitation_now
FROM {PRECOMPUTE_TABLE_NAME} p
JOIN imerg_window ilab
  ON ilab.grid_number = p.grid_number
 AND ilab.t = p.local_forecast_time
{feature_join}
LEFT JOIN imerg_neighbor nb
  ON nb.grid_number = p.grid_number
 AND nb.t = {neighbor_time}
WHERE p.pressure_msl_change_6h IS NOT NULL
  AND p.precipitation_lag_6h IS NOT NULL
  AND p.precipitation_sum_past_24h IS NOT NULL
  AND p.cloud_cover_lag_6h IS NOT NULL
  AND p.humidity_lag_6h IS NOT NULL
  AND p.wind_speed_lag_3h IS NOT NULL
  AND p.precipitation_next_{max(HORIZONS)}h IS NOT NULL
  AND p.neighbor_count > 0
  AND ilab.imerg_precipitation_next_{max(HORIZONS)}h IS NOT NULL
  AND {feature_alias}.imerg_history_hours_24 = 24
ORDER BY p.local_forecast_time, p.grid_number
"""


def rain_3class(precipitation_mm):
    """0 = no_rain, 1 = light, 2 = moderate_or_heavy. NaN-safe, returns int8."""
    values = np.asarray(precipitation_mm, dtype="float64")
    classes = np.zeros(len(values), dtype="int8")
    classes[values >= INTENSITY_THRESHOLDS_MM["rain_min"]] = 1
    classes[values >= INTENSITY_THRESHOLDS_MM["moderate_or_heavy_min"]] = 2
    return classes


def add_targets(data, label_prefix):
    """Build the 3-class and two stage targets from the chosen label source."""
    for horizon in HORIZONS:
        source = f"{label_prefix}_precipitation_next_{horizon}h"
        final_target = f"rain_3class_next_{horizon}h"
        data[final_target] = rain_3class(data[source])
        data[f"strong_vs_rest_next_{horizon}h"] = (data[final_target] == 2).astype("int8")
        data[f"light_vs_no_rain_next_{horizon}h"] = (data[final_target] == 1).astype("int8")
    return data


def read_training_data(sample_rows=None, use_cache=USE_CACHE):
    if use_cache and CACHE_PATH.exists():
        print(f"Loading cached join from {CACHE_PATH}")
        data = pd.read_pickle(CACHE_PATH)
    else:
        query = build_query(IMERG_FEATURE_LAG_HOURS, IMERG_RUN_TYPES)
        started = time.time()
        with connect() as conn:
            data = pd.read_sql_query(query, conn, parse_dates=["forecast_time"])
        print(f"Query returned {len(data):,} rows in {time.time() - started:.1f}s")
        # ~2.45M rows x ~100 numeric columns is around 1.9 GB in float64; float32 halves it
        # and matches the dtype LightGBM is fed anyway.
        float_columns = data.select_dtypes("float64").columns
        data[float_columns] = data[float_columns].astype("float32")
        CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        data.to_pickle(CACHE_PATH)
        print(f"Cached to {CACHE_PATH}")

    if sample_rows:
        data = data.sample(n=int(sample_rows), random_state=RANDOM_STATE).sort_values(
            ["forecast_time", "grid_number"]
        ).reset_index(drop=True)

    return add_targets(data, LABEL_PREFIX)

In [ ]:

df = read_training_data(SAMPLE_ROWS)
print(df.shape)
print(df["forecast_time"].min(), "to", df["forecast_time"].max())
print(df["imerg_run_type"].value_counts().to_dict())
display(df.head())

required_columns = sorted(set(FEATURE_COLUMNS + FUTURE_PRECIP_COLUMNS + TARGET_COLUMNS))
missing_counts = df[required_columns].isna().sum().sort_values(ascending=False)
display(missing_counts[missing_counts > 0].rename("missing_rows"))
if missing_counts.max() == 0:
    print("No missing values in features, future precipitation or targets.")


## 5. Label Source Audit: Open-Meteo Versus IMERG

This section is the justification for the V5 label change. It reproduces, from the loaded
data, the disagreement between the two precipitation series. Read the diurnal chart first:
the phase offset is the clearest evidence that V4 was fitting model timing rather than
observed timing.

In [ ]:

agreement = pd.DataFrame({
    "metric": [
        "rows",
        "pearson_correlation",
        "open_meteo_mean_mm",
        "imerg_mean_mm",
        "open_meteo_rain_rate",
        "imerg_rain_rate",
        "open_meteo_strong_rate",
        "imerg_strong_rate",
    ],
    "value": [
        len(df),
        df["precipitation"].corr(df["imerg_precipitation"]),
        df["precipitation"].mean(),
        df["imerg_precipitation"].mean(),
        (df["precipitation"] >= INTENSITY_THRESHOLDS_MM["rain_min"]).mean(),
        (df["imerg_precipitation"] >= INTENSITY_THRESHOLDS_MM["rain_min"]).mean(),
        (df["precipitation"] >= INTENSITY_THRESHOLDS_MM["moderate_or_heavy_min"]).mean(),
        (df["imerg_precipitation"] >= INTENSITY_THRESHOLDS_MM["moderate_or_heavy_min"]).mean(),
    ],
})
display(agreement)

om_class_now = rain_3class(df["precipitation"])
imerg_class_now = rain_3class(df["imerg_precipitation"])
cross = pd.crosstab(
    pd.Series(om_class_now, name="open_meteo_class").map(CLASS_LABELS),
    pd.Series(imerg_class_now, name="imerg_class").map(CLASS_LABELS),
    normalize="all",
)
print("Class agreement at the same hour (fraction of all rows):")
display(cross)
print(f"Exact class agreement: {(om_class_now == imerg_class_now).mean():.4f}")

strong_om = om_class_now == 2
strong_imerg = imerg_class_now == 2
print(
    "Of hours Open-Meteo calls moderate_or_heavy, "
    f"{(strong_imerg & strong_om).sum() / max(strong_om.sum(), 1):.3f} are also moderate_or_heavy in IMERG."
)

In [ ]:

diurnal = (
    df.assign(hour=df["forecast_time"].dt.hour)
      .groupby("hour")[["precipitation", "imerg_precipitation"]]
      .mean()
      .rename(columns={"precipitation": "open_meteo", "imerg_precipitation": "imerg"})
)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
diurnal.plot(ax=axes[0], marker="o")
axes[0].set_title("Mean hourly precipitation by local hour")
axes[0].set_xlabel("Local hour")
axes[0].set_ylabel("mm")
axes[0].axvline(diurnal["open_meteo"].idxmax(), color="tab:blue", linestyle="--", alpha=0.5)
axes[0].axvline(diurnal["imerg"].idxmax(), color="tab:orange", linestyle="--", alpha=0.5)

monthly = (
    df.assign(month=df["forecast_time"].dt.month)
      .groupby("month")[["precipitation", "imerg_precipitation"]]
      .mean()
      .rename(columns={"precipitation": "open_meteo", "imerg_precipitation": "imerg"})
)
monthly.plot(ax=axes[1], marker="o")
axes[1].set_title("Mean hourly precipitation by month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("mm")
plt.tight_layout()
plt.show()

print(f"Open-Meteo diurnal peak hour: {diurnal['open_meteo'].idxmax()}")
print(f"IMERG diurnal peak hour     : {diurnal['imerg'].idxmax()}")
print(f"Phase offset (hours)        : {diurnal['imerg'].idxmax() - diurnal['open_meteo'].idxmax()}")

## 6. Class Balance Under The Selected Label Source

In [ ]:

balance_rows = []
for horizon in HORIZONS:
    target = f"rain_3class_next_{horizon}h"
    counts = df[target].value_counts().reindex(CLASS_ORDER, fill_value=0)
    for class_id, rows in counts.items():
        balance_rows.append({
            "horizon_h": horizon,
            "label_source": LABEL_SOURCE,
            "target_column": target,
            "class_id": int(class_id),
            "class_label": CLASS_LABELS[int(class_id)],
            "rows": int(rows),
            "class_rate": float(rows / len(df)),
        })

target_balance = pd.DataFrame(balance_rows)
display(target_balance)

# Same table under the other label source, so the V4 comparison is explicit.
other_source = "open_meteo" if LABEL_SOURCE == "imerg" else "imerg"
other_prefix = LABEL_PREFIXES[other_source]
comparison_rows = []
for horizon in HORIZONS:
    for source_name, prefix in [(LABEL_SOURCE, LABEL_PREFIX), (other_source, other_prefix)]:
        classes = rain_3class(df[f"{prefix}_precipitation_next_{horizon}h"])
        for class_id in CLASS_ORDER:
            comparison_rows.append({
                "horizon_h": horizon,
                "label_source": source_name,
                "class_label": CLASS_LABELS[class_id],
                "class_rate": float((classes == class_id).mean()),
            })
label_source_balance = pd.DataFrame(comparison_rows)

sns.catplot(
    data=label_source_balance,
    x="horizon_h", y="class_rate", hue="class_label", col="label_source",
    kind="bar", height=4.5, aspect=1.3,
)
plt.subplots_adjust(top=0.86)
plt.suptitle("Class rate by horizon and label source")
plt.show()

## 7. Chronological Split

In [ ]:

def add_time_split(data, train_fraction=0.70, validation_fraction=0.15):
    unique_times = np.array(sorted(data["forecast_time"].unique()))
    train_end = unique_times[int(len(unique_times) * train_fraction)]
    validation_end = unique_times[int(len(unique_times) * (train_fraction + validation_fraction))]
    out = data.copy()
    out["split"] = "test"
    out.loc[out["forecast_time"] < train_end, "split"] = "train"
    out.loc[
        (out["forecast_time"] >= train_end) & (out["forecast_time"] < validation_end),
        "split",
    ] = "validation"
    return out, train_end, validation_end


model_df, train_end, validation_end = add_time_split(df, TRAIN_FRACTION, VALIDATION_FRACTION)
print(f"Train before      : {train_end}")
print(f"Validation before : {validation_end}")

split_summary = (
    model_df.groupby("split")
    .agg(
        rows=("grid_number", "size"),
        first_time=("forecast_time", "min"),
        last_time=("forecast_time", "max"),
        strong_rate_next_1h=("strong_vs_rest_next_1h", "mean"),
    )
    .reindex(["train", "validation", "test"])
)
display(split_summary)

# IMERG run type by split: the provisional (Late Run) period backs the recent months.
display(pd.crosstab(model_df["split"], model_df["imerg_run_type"]).reindex(
    ["train", "validation", "test"]
))

train_df = model_df[model_df["split"] == "train"]
validation_df = model_df[model_df["split"] == "validation"]
test_df = model_df[model_df["split"] == "test"]

x_train = train_df[FEATURE_COLUMNS].astype("float32")
x_validation = validation_df[FEATURE_COLUMNS].astype("float32")
x_test = test_df[FEATURE_COLUMNS].astype("float32")


## 8. Persistence Baseline

V4 reported no baseline, so there was no way to tell how much of its score came from the
model rather than from rain simply continuing. This classifies `t+h` using the observed
IMERG class at `t`. Every trained model below should beat it; if it does not at a given
horizon, that horizon is not learning anything.

In [ ]:

def baseline_rows(reference_column):
    rows = []
    reference_class = rain_3class(model_df[reference_column])
    for split_name in ["validation", "test"]:
        mask = (model_df["split"] == split_name).to_numpy()
        for horizon in HORIZONS:
            y_true = model_df.loc[mask, f"rain_3class_next_{horizon}h"].to_numpy()
            y_pred = reference_class[mask]
            rows.append({
                "model": "persistence",
                "horizon_h": horizon,
                "split": split_name,
                "rows": int(mask.sum()),
                "accuracy": float(accuracy_score(y_true, y_pred)),
                "macro_f1": float(f1_score(y_true, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0)),
                "moderate_or_heavy_recall": float(
                    recall_score((y_true == 2).astype("int8"), (y_pred == 2).astype("int8"), zero_division=0)
                ),
                "moderate_or_heavy_precision": float(
                    precision_score((y_true == 2).astype("int8"), (y_pred == 2).astype("int8"), zero_division=0)
                ),
            })
    return rows


baseline_reference = "imerg_precipitation" if LABEL_SOURCE == "imerg" else "precipitation"
baseline_metrics = pd.DataFrame(baseline_rows(baseline_reference))
display(baseline_metrics.sort_values(["split", "horizon_h"]))


## 9. Strong-First Evaluation And Constrained Threshold Tuning

The hierarchy is unchanged from V4, but two things in the threshold search are fixed.

**Unreachable-constraint fallback.** V4 ranked candidates by strong recall whenever no
threshold pair reached `TARGET_VALIDATION_STRONG_RECALL`. Because recall increases
monotonically as the threshold falls, that rule always returned the smallest threshold in
the grid, labelling nearly every row `moderate_or_heavy`. On a short-window trial of this
notebook that fallback produced a strong precision of 0.017 and a macro F1 of 0.15 at +3h.
V5 falls back to the best strong F1, and `meets_recall_constraint` still reports the miss.

**Grid range.** The strong grid now runs to 0.95 rather than 0.61, so high-precision
operating points are actually reachable.

In [ ]:

def predict_three_class(strong_probability, light_probability, strong_threshold, light_threshold):
    predictions = np.zeros(len(strong_probability), dtype="int8")
    strong_mask = strong_probability >= strong_threshold
    predictions[strong_mask] = 2
    predictions[(~strong_mask) & (light_probability >= light_threshold)] = 1
    return predictions


def class_metric_rows(y_true, y_pred, strong_probability, light_probability, model_name, horizon,
                      split, strong_threshold, light_threshold, label_source):
    rows = []
    for class_id, class_label in CLASS_LABELS.items():
        binary_true = (y_true == class_id).astype("int8")
        binary_pred = (y_pred == class_id).astype("int8")
        rows.append({
            "model": model_name,
            "target_type": "exact_hour_3class_strong_first_recall_constrained",
            "label_source": label_source,
            "feature_set": FEATURE_SET_NAME,
            "horizon_h": horizon,
            "split": split,
            "class_id": class_id,
            "class_label": class_label,
            "rows": int(len(y_true)),
            "class_rate": float(binary_true.mean()),
            "predicted_class_rate": float(binary_pred.mean()),
            "precision": float(precision_score(binary_true, binary_pred, zero_division=0)),
            "recall": float(recall_score(binary_true, binary_pred, zero_division=0)),
            "f1": float(f1_score(binary_true, binary_pred, zero_division=0)),
            "strong_threshold": float(strong_threshold),
            "light_threshold": float(light_threshold),
            "mean_strong_probability": float(strong_probability.mean()),
            "mean_light_probability": float(light_probability.mean()),
        })
    return rows


def evaluate_predictions(y_true, y_pred, strong_probability, light_probability, model_name,
                         horizon, split, strong_threshold, light_threshold,
                         label_source, subgroup="all"):
    strong_true = (y_true == 2).astype("int8")
    rain_true = (y_true > 0).astype("int8")
    rain_pred = (y_pred > 0).astype("int8")
    base = {
        "model": model_name,
        "target_type": "exact_hour_3class_strong_first_recall_constrained",
        "label_source": label_source,
        "feature_set": FEATURE_SET_NAME,
        "subgroup": subgroup,
        "horizon_h": horizon,
        "split": split,
        "rows": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=CLASS_ORDER, average="weighted", zero_division=0)),
        "rain_precision": float(precision_score(rain_true, rain_pred, zero_division=0)),
        "rain_recall": float(recall_score(rain_true, rain_pred, zero_division=0)),
        "strong_average_precision": float(average_precision_score(strong_true, strong_probability))
            if strong_true.sum() > 0 else float("nan"),
        "strong_threshold": float(strong_threshold),
        "light_threshold": float(light_threshold),
    }
    class_rows = class_metric_rows(
        y_true, y_pred, strong_probability, light_probability, model_name, horizon, split,
        strong_threshold, light_threshold, label_source,
    )
    confusion = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=CLASS_ORDER),
        index=[f"actual_{CLASS_LABELS[i]}" for i in CLASS_ORDER],
        columns=[f"predicted_{CLASS_LABELS[i]}" for i in CLASS_ORDER],
    ).reset_index(names="actual_class")
    confusion.insert(0, "horizon_h", horizon)
    confusion.insert(1, "split", split)
    confusion.insert(2, "label_source", label_source)
    confusion.insert(3, "subgroup", subgroup)
    confusion.insert(4, "strong_threshold", float(strong_threshold))
    confusion.insert(5, "light_threshold", float(light_threshold))
    return base, class_rows, confusion


def tune_thresholds(y_true, strong_probability, light_probability):
    """Hard constraint on strong recall first, then maximise strong precision and macro F1.

    Fallback fix versus V4: when no threshold pair can reach the recall target, V4 ranked
    the remaining candidates by strong recall. That always selects the lowest threshold in
    the grid, which labels almost every row moderate_or_heavy and destroys precision and
    macro F1 at exactly the horizons where the constraint is hardest to meet. V5 falls back
    to the best strong F1 instead, which is a usable operating point, and still records
    `meets_recall_constraint = False` so the shortfall stays visible.
    """
    rows = []
    best = None
    best_key = None
    strong_true = (y_true == 2).astype("int8")
    for strong_threshold in STRONG_PROBABILITY_THRESHOLDS_TO_TEST:
        for light_threshold in LIGHT_PROBABILITY_THRESHOLDS_TO_TEST:
            y_pred = predict_three_class(
                strong_probability, light_probability, strong_threshold, light_threshold
            )
            strong_pred = (y_pred == 2).astype("int8")
            strong_recall = recall_score(strong_true, strong_pred, zero_division=0)
            strong_precision = precision_score(strong_true, strong_pred, zero_division=0)
            meets = bool(strong_recall >= TARGET_VALIDATION_STRONG_RECALL)
            candidate = {
                "strong_threshold": float(strong_threshold),
                "light_threshold": float(light_threshold),
                "meets_recall_constraint": meets,
                "macro_f1": float(f1_score(y_true, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0)),
                "moderate_or_heavy_precision": float(strong_precision),
                "moderate_or_heavy_recall": float(strong_recall),
                "moderate_or_heavy_f1": float(f1_score(strong_true, strong_pred, zero_division=0)),
            }
            rows.append(candidate)
            candidate_key = (
                int(meets),
                strong_precision if meets else candidate["moderate_or_heavy_f1"],
                candidate["macro_f1"],
                strong_recall,
            )
            if best_key is None or candidate_key > best_key:
                best, best_key = candidate, candidate_key
    return best.copy(), pd.DataFrame(rows)


## 10. Train Strong-First Models

Two changes from V4 beyond the inputs and labels:

- Boosting rounds are chosen by early stopping on the validation split instead of being
  fixed at 500, so each horizon gets the depth it needs and over-fitting is bounded.
- The fitted tree count is recorded per stage so the search can be audited.

In [ ]:

try:
    from lightgbm import LGBMClassifier, early_stopping, log_evaluation
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install LightGBM first: %pip install lightgbm") from exc

LGBM_PARAMS = dict(
    objective="binary",
    class_weight="balanced",
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    num_leaves=63,
    min_child_samples=80,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
FIT_CALLBACKS = [early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(0)]

strong_models = {}
light_models = {}
threshold_rows = []
threshold_search_tables = []
metric_rows = []
class_metric_rows_all = []
confusion_tables = []
feature_importance_rows = []
training_log_rows = []

for horizon in HORIZONS:
    final_target = f"rain_3class_next_{horizon}h"
    strong_target = f"strong_vs_rest_next_{horizon}h"
    light_target = f"light_vs_no_rain_next_{horizon}h"
    print(f"Training V5 strong-first models for exact +{horizon}h...", flush=True)

    y_final_validation = validation_df[final_target].astype("int8").to_numpy()
    y_final_test = test_df[final_target].astype("int8").to_numpy()

    strong_model = LGBMClassifier(**LGBM_PARAMS)
    strong_model.fit(
        x_train,
        train_df[strong_target].astype("int8"),
        eval_set=[(x_validation, validation_df[strong_target].astype("int8"))],
        eval_metric="binary_logloss",
        callbacks=FIT_CALLBACKS,
    )
    strong_models[horizon] = strong_model

    nonstrong_train = train_df[final_target] != 2
    nonstrong_validation = validation_df[final_target] != 2
    light_model = LGBMClassifier(**LGBM_PARAMS)
    light_model.fit(
        train_df.loc[nonstrong_train, FEATURE_COLUMNS].astype("float32"),
        train_df.loc[nonstrong_train, light_target].astype("int8"),
        eval_set=[(
            validation_df.loc[nonstrong_validation, FEATURE_COLUMNS].astype("float32"),
            validation_df.loc[nonstrong_validation, light_target].astype("int8"),
        )],
        eval_metric="binary_logloss",
        callbacks=FIT_CALLBACKS,
    )
    light_models[horizon] = light_model

    training_log_rows.append({
        "horizon_h": horizon,
        "strong_best_iteration": int(strong_model.best_iteration_ or N_ESTIMATORS),
        "light_best_iteration": int(light_model.best_iteration_ or N_ESTIMATORS),
        "n_estimators_cap": N_ESTIMATORS,
        "train_rows": int(len(train_df)),
        "strong_train_rate": float(train_df[strong_target].mean()),
    })

    strong_validation_probability = strong_model.predict_proba(x_validation)[:, 1]
    light_validation_probability = light_model.predict_proba(x_validation)[:, 1]
    best_thresholds, threshold_search = tune_thresholds(
        y_final_validation, strong_validation_probability, light_validation_probability
    )
    best_thresholds["horizon_h"] = horizon
    threshold_rows.append(best_thresholds)
    threshold_search["horizon_h"] = horizon
    threshold_search_tables.append(threshold_search)

    for model_stage, model in [("strong_vs_rest", strong_model), ("light_vs_no_rain", light_model)]:
        for feature, importance in zip(FEATURE_COLUMNS, np.asarray(model.feature_importances_)):
            feature_importance_rows.append({
                "model": "lightgbm",
                "model_stage": model_stage,
                "horizon_h": horizon,
                "feature": feature,
                "importance": float(importance),
                "feature_family": (
                    "imerg" if feature in IMERG_FEATURE_COLUMNS
                    else "om_neighbor" if feature in NEIGHBOR_FEATURE_COLUMNS
                    else "om_baseline"
                ),
            })

    for split_name, split_frame, x_split, y_final in [
        ("validation", validation_df, x_validation, y_final_validation),
        ("test", test_df, x_test, y_final_test),
    ]:
        strong_probability = strong_model.predict_proba(x_split)[:, 1]
        light_probability = light_model.predict_proba(x_split)[:, 1]
        y_pred = predict_three_class(
            strong_probability,
            light_probability,
            best_thresholds["strong_threshold"],
            best_thresholds["light_threshold"],
        )
        metrics, class_rows, confusion = evaluate_predictions(
            y_final, y_pred, strong_probability, light_probability,
            "lightgbm_strong_first", horizon, split_name,
            best_thresholds["strong_threshold"], best_thresholds["light_threshold"],
            LABEL_SOURCE,
        )
        metric_rows.append(metrics)
        class_metric_rows_all.extend(class_rows)
        confusion_tables.append(confusion)

        # Same predictions, scored against the other label source.
        if EVALUATE_BOTH_LABEL_SOURCES:
            other_true = rain_3class(
                split_frame[f"{other_prefix}_precipitation_next_{horizon}h"]
            )
            other_metrics, other_class_rows, other_confusion = evaluate_predictions(
                other_true, y_pred, strong_probability, light_probability,
                "lightgbm_strong_first", horizon, split_name,
                best_thresholds["strong_threshold"], best_thresholds["light_threshold"],
                other_source,
            )
            metric_rows.append(other_metrics)
            class_metric_rows_all.extend(other_class_rows)
            confusion_tables.append(other_confusion)

        # Test metrics broken out by IMERG run type (Final versus Late Run provenance).
        if split_name == "test":
            run_types = split_frame["imerg_run_type"].to_numpy()
            for run_type in pd.unique(run_types):
                mask = run_types == run_type
                if mask.sum() < 1000:
                    continue
                subgroup_metrics, _, _ = evaluate_predictions(
                    y_final[mask], y_pred[mask], strong_probability[mask], light_probability[mask],
                    "lightgbm_strong_first", horizon, split_name,
                    best_thresholds["strong_threshold"], best_thresholds["light_threshold"],
                    LABEL_SOURCE, subgroup=f"run_type={run_type}",
                )
                metric_rows.append(subgroup_metrics)

thresholds = pd.DataFrame(threshold_rows)
threshold_search_results = pd.concat(threshold_search_tables, ignore_index=True)
all_metrics = pd.DataFrame(metric_rows)
class_metrics = pd.DataFrame(class_metric_rows_all)
confusion_results = pd.concat(confusion_tables, ignore_index=True)
feature_importance = pd.DataFrame(feature_importance_rows)
training_log = pd.DataFrame(training_log_rows)

primary_metrics = all_metrics[
    (all_metrics["label_source"] == LABEL_SOURCE) & (all_metrics["subgroup"] == "all")
]

display(training_log)
display(thresholds.sort_values("horizon_h"))
display(primary_metrics.sort_values(["horizon_h", "split"]))
display(
    class_metrics[
        (class_metrics["split"] == "test") & (class_metrics["label_source"] == LABEL_SOURCE)
    ].sort_values(["horizon_h", "class_id"])
)

## 11. Results, Baseline Comparison And Provenance Check

In [ ]:

test_primary = primary_metrics[primary_metrics["split"] == "test"]
test_strong = class_metrics[
    (class_metrics["split"] == "test")
    & (class_metrics["label_source"] == LABEL_SOURCE)
    & (class_metrics["class_id"] == 2)
]
baseline_test = baseline_metrics[baseline_metrics["split"] == "test"]

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

sns.lineplot(data=test_primary, x="horizon_h", y="macro_f1", marker="o", label="V5 model", ax=axes[0])
sns.lineplot(data=baseline_test, x="horizon_h", y="macro_f1", marker="s", label="persistence", ax=axes[0])
axes[0].set_title(f"Test macro F1 ({LABEL_SOURCE} labels)")
axes[0].set_ylabel("Macro F1")

sns.lineplot(data=test_strong, x="horizon_h", y="recall", marker="o", label="V5 model", ax=axes[1])
sns.lineplot(
    data=baseline_test, x="horizon_h", y="moderate_or_heavy_recall", marker="s",
    label="persistence", ax=axes[1],
)
axes[1].axhline(0.65, color="red", linestyle="--", label="65% target")
axes[1].set_title("Test moderate/heavy recall")
axes[1].set_ylabel("Recall")
axes[1].legend()

sns.lineplot(data=test_strong, x="horizon_h", y="precision", marker="o", label="V5 model", ax=axes[2])
sns.lineplot(
    data=baseline_test, x="horizon_h", y="moderate_or_heavy_precision", marker="s",
    label="persistence", ax=axes[2],
)
axes[2].set_title("Test moderate/heavy precision")
axes[2].set_ylabel("Precision")

for ax in axes:
    ax.set_xlabel("Forecast horizon: exact hour t+h")
plt.tight_layout()
plt.show()

sns.catplot(
    data=class_metrics[
        (class_metrics["split"] == "test") & (class_metrics["label_source"] == LABEL_SOURCE)
    ],
    x="horizon_h", y="f1", hue="class_label", kind="bar", height=5, aspect=1.8,
)
plt.title(f"BKK 3-Class V5 test F1 by class ({LABEL_SOURCE} labels)")
plt.xlabel("Forecast horizon: exact hour t+h")
plt.ylabel("F1")
plt.show()

In [ ]:

# Validation-to-test recall decline, the failure mode V4 was built around.
recall_drift = (
    class_metrics[
        (class_metrics["class_id"] == 2) & (class_metrics["label_source"] == LABEL_SOURCE)
    ]
    .pivot_table(index="horizon_h", columns="split", values=["recall", "precision"])
)
recall_drift[("recall", "drop")] = recall_drift[("recall", "validation")] - recall_drift[("recall", "test")]
print("Moderate/heavy recall, validation versus test:")
display(recall_drift)

# Does the model still hold up when scored against the Open-Meteo labels V4 used?
if EVALUATE_BOTH_LABEL_SOURCES:
    print("\nSame V5 predictions scored against both label sources (test split):")
    display(
        all_metrics[(all_metrics["split"] == "test") & (all_metrics["subgroup"] == "all")]
        .pivot_table(index="horizon_h", columns="label_source",
                     values=["macro_f1", "balanced_accuracy", "rain_recall"])
    )

# Provenance: Final Run versus Late Run months inside the test split.
provenance = all_metrics[
    (all_metrics["split"] == "test") & (all_metrics["subgroup"] != "all")
]
if not provenance.empty:
    print("\nTest metrics by IMERG run type. Large gaps here mean the score depends on")
    print("provisional Late Run data that will be revised when Final Run catches up.")
    display(
        provenance.pivot_table(index="horizon_h", columns="subgroup",
                               values=["macro_f1", "strong_average_precision"])
    )

In [ ]:

# Where the signal actually comes from: importance share by feature family.
family_share = (
    feature_importance.groupby(["model_stage", "horizon_h", "feature_family"])["importance"]
    .sum()
    .reset_index()
)
family_share["share"] = family_share.groupby(["model_stage", "horizon_h"])["importance"].transform(
    lambda s: s / s.sum()
)
sns.catplot(
    data=family_share, x="horizon_h", y="share", hue="feature_family", col="model_stage",
    kind="bar", height=4.5, aspect=1.2,
)
plt.subplots_adjust(top=0.86)
plt.suptitle("LightGBM importance share by feature family")
plt.show()

display(
    feature_importance.sort_values(
        ["model_stage", "horizon_h", "importance"], ascending=[True, True, False]
    )
    .groupby(["model_stage", "horizon_h"])
    .head(12)
)


## 12. Compare Against V4

Loads `bkk_3_class_v4_metrics.csv` if the V4 notebook has been run. Note what is and is not
comparable: V4 was scored against Open-Meteo labels, so the honest comparison is the
`label_source == "open_meteo"` rows of V5, which use the identical target definition. The
`imerg` rows answer a different and harder question, namely predicting observed rain.

In [ ]:

v4_metrics_path = PROJECT_ROOT / "ML_Model_V2" / "trained_models" / "BKK_3_Class_V4" / "bkk_3_class_v4_metrics.csv"
if v4_metrics_path.exists():
    v4_metrics = pd.read_csv(v4_metrics_path)
    v4_test = (
        v4_metrics[v4_metrics["split"] == "test"]
        .set_index("horizon_h")[["macro_f1", "balanced_accuracy", "rain_recall"]]
        .add_prefix("v4_")
    )
    v5_test_om = (
        all_metrics[
            (all_metrics["split"] == "test")
            & (all_metrics["subgroup"] == "all")
            & (all_metrics["label_source"] == "open_meteo")
        ]
        .set_index("horizon_h")[["macro_f1", "balanced_accuracy", "rain_recall"]]
        .add_prefix("v5_om_labels_")
    )
    v5_test_imerg = (
        all_metrics[
            (all_metrics["split"] == "test")
            & (all_metrics["subgroup"] == "all")
            & (all_metrics["label_source"] == "imerg")
        ]
        .set_index("horizon_h")[["macro_f1", "balanced_accuracy", "rain_recall"]]
        .add_prefix("v5_imerg_labels_")
    )
    comparison = v4_test.join(v5_test_om, how="outer").join(v5_test_imerg, how="outer")
    display(comparison)
else:
    print(f"V4 metrics not found at {v4_metrics_path}; run BKK_3_Class_V4.ipynb to enable this comparison.")

## 13. Save BKK 3-Class V5 Models And Results

In [ ]:

MODEL_DIR.mkdir(parents=True, exist_ok=True)

for horizon, model in strong_models.items():
    joblib.dump(model, MODEL_DIR / f"bkk_3_class_v5_exact_next_{horizon}h_strong_vs_rest_lightgbm.joblib")

for horizon, model in light_models.items():
    joblib.dump(model, MODEL_DIR / f"bkk_3_class_v5_exact_next_{horizon}h_light_vs_no_rain_lightgbm.joblib")

target_balance.to_csv(MODEL_DIR / "bkk_3_class_v5_target_balance.csv", index=False)
label_source_balance.to_csv(MODEL_DIR / "bkk_3_class_v5_label_source_balance.csv", index=False)
agreement.to_csv(MODEL_DIR / "bkk_3_class_v5_label_source_agreement.csv", index=False)
baseline_metrics.to_csv(MODEL_DIR / "bkk_3_class_v5_persistence_baseline.csv", index=False)
thresholds.to_csv(MODEL_DIR / "bkk_3_class_v5_thresholds.csv", index=False)
threshold_search_results.to_csv(MODEL_DIR / "bkk_3_class_v5_threshold_search.csv", index=False)
all_metrics.to_csv(MODEL_DIR / "bkk_3_class_v5_metrics.csv", index=False)
class_metrics.to_csv(MODEL_DIR / "bkk_3_class_v5_class_metrics.csv", index=False)
confusion_results.to_csv(MODEL_DIR / "bkk_3_class_v5_confusion_matrices.csv", index=False)
feature_importance.to_csv(MODEL_DIR / "bkk_3_class_v5_lightgbm_feature_importance.csv", index=False)
training_log.to_csv(MODEL_DIR / "bkk_3_class_v5_training_log.csv", index=False)

metadata = {
    "table": TABLE_NAME,
    "precompute_table": PRECOMPUTE_TABLE_NAME,
    "imerg_table": IMERG_TABLE_NAME,
    "target_type": "rain_3class_exact_at_t_plus_h_strong_first_recall_constrained",
    "label_source": LABEL_SOURCE,
    "label_column_prefix": LABEL_PREFIX,
    "imerg_run_types": list(IMERG_RUN_TYPES),
    "use_imerg_history_features": USE_IMERG_HISTORY_FEATURES,
    "imerg_feature_lag_hours": IMERG_FEATURE_LAG_HOURS,
    "operational_note": (
        "IMERG Final Run lags roughly 3.5 months. With imerg_feature_lag_hours = 0 and "
        "use_imerg_history_features = true these metrics are a research upper bound, not a "
        "real-time deployable score. Re-run with use_imerg_history_features = false, or with "
        "imerg_feature_lag_hours set to the latency of the intended IMERG Early Run feed."
    ),
    "horizons": HORIZONS,
    "intensity_thresholds_mm": INTENSITY_THRESHOLDS_MM,
    "class_labels": CLASS_LABELS,
    "model": "strong_first_two_stage_lightgbm",
    "stage_1": "strong_vs_rest",
    "stage_2": "light_vs_no_rain",
    "lgbm_params": {k: v for k, v in LGBM_PARAMS.items()},
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "target_validation_strong_recall": TARGET_VALIDATION_STRONG_RECALL,
    "feature_set": FEATURE_SET_NAME,
    "feature_columns": FEATURE_COLUMNS,
    "open_meteo_feature_columns": OPEN_METEO_FEATURE_COLUMNS,
    "imerg_feature_columns": IMERG_FEATURE_COLUMNS,
    "future_precip_columns": FUTURE_PRECIP_COLUMNS,
    "final_target_columns": FINAL_TARGET_COLUMNS,
    "strong_target_columns": STRONG_TARGET_COLUMNS,
    "light_target_columns": LIGHT_TARGET_COLUMNS,
    "train_end_exclusive": str(train_end),
    "validation_end_exclusive": str(validation_end),
    "rows": int(len(df)),
    "sample_rows": SAMPLE_ROWS,
    "model_dir": str(MODEL_DIR),
    "training_log": training_log.to_dict(orient="records"),
    "thresholds": thresholds.to_dict(orient="records"),
    "target_balance": target_balance.to_dict(orient="records"),
    "persistence_baseline": baseline_metrics.to_dict(orient="records"),
    "metrics": all_metrics.to_dict(orient="records"),
    "class_metrics": class_metrics.to_dict(orient="records"),
}
(MODEL_DIR / "bkk_3_class_v5_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved BKK 3-Class V5 models and result CSVs to {MODEL_DIR}")


## 14. Suggested Next Runs

1. **Deployability check.** Set `USE_IMERG_HISTORY_FEATURES = False` and re-run. The gap
   against this run is the value the satellite history is adding, and the resulting model
   is the one that could run in real time against Open-Meteo alone.
2. **Early Run emulation.** Set `IMERG_FEATURE_LAG_HOURS = 4` with the history features on.
   This approximates what a live IMERG Early Run feed would deliver.
3. **V4 reproduction.** Set `LABEL_SOURCE = "open_meteo"` and
   `USE_IMERG_HISTORY_FEATURES = False` to reproduce V4 inside this notebook and confirm
   the harness matches before trusting any delta.
4. **Provenance.** Once IMERG Final Run catches up past 2025-10-01, re-run the backfill and
   set `IMERG_RUN_TYPES = ("permanent", "Final")` to drop provisional data from the test
   split.
5. **Feature promotion.** If the IMERG features hold up, move the section 4 query into
   `OM_Script/precompute_om_bkk_features.py` (or a new `precompute_imerg_bkk_features.py`)
   so the join is materialised once instead of recomputed per notebook run.